In [1]:
import json
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from cleanvision import Imagelab
from datasets import concatenate_datasets

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from scripts.load_dataset import DROP_FILE, load_raw  # erst nach sys.path.insert importierbar

out = ROOT / "notebooks/results/station1/exploration"
out.mkdir(parents=True, exist_ok=True)

train_hf, val_hf, test_hf = load_raw(seed=67)
class_names = train_hf.features["label"].names

print(train_hf.features["label"])

counts = train_hf.select_columns(["label"]).to_pandas()["label"].value_counts()
counts.index = [class_names[i] for i in counts.index]
print(counts.describe())                       # Bilder pro Klasse: min / max / mean
print("max/min:", counts.max() / counts.min())
print(counts.head(5))                          # häufigste Klassen
print(counts.tail(5))                          # seltenste Klassen

sizes, modes = [], Counter()
for ex in train_hf:
    sizes.append(ex["image"].size)
    modes[ex["image"].mode] += 1

w, h = np.array(sizes).T
print(modes)                                   # z. B. Counter({'RGB': 8120, 'L': 24})
print("Breite min/median/max:", w.min(), np.median(w), w.max())
print("Höhe   min/median/max:", h.min(), np.median(h), h.max())
print("Seitenverhältnis > 2:1:", (w / h > 2).sum())

def show_grid(subset, path):
    fig, axes = plt.subplots(5, 5, figsize=(15, 15))
    for ax in axes.flat:
        ax.axis("off")
    for ax, ex in zip(axes.flat, subset):
        ax.imshow(ex["image"].convert("RGB"))
        ax.set_title(class_names[ex["label"]], fontsize=7)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)

show_grid(train_hf.shuffle(seed=0).select(range(25)), out / "random.png")

k = 0                                          # Klassen-Index, Name: class_names[k]
one_class = train_hf.filter(lambda label: label == k, input_columns="label")
show_grid(one_class.select(range(min(25, len(one_class)))), out / f"class_{k}.png")

imagelab = Imagelab(hf_dataset=train_hf, image_key="image")
imagelab.find_issues()
imagelab.report()

both = concatenate_datasets([train_hf, val_hf, test_hf])  # Indizes: [Train | Val | Test]
lab = Imagelab(hf_dataset=both, image_key="image")
lab.find_issues(issue_types={"exact_duplicates": {}, "near_duplicates": {}})

n_train = len(train_hf)
n_tv = n_train + len(val_hf)              # Grenze Train+Val | Test
cross = [s for key in ["exact_duplicates", "near_duplicates"]
         for s in lab.info[key]["sets"]
         if min(s) < n_tv <= max(s)]           # Gruppe enthält Train/Val- und Test-Index
print(len(cross), "Duplikat-Gruppen über Train/Val und Test hinweg")

drop_train, drop_val = set(), set()

def mark(i):
    i = int(i)
    if i < n_train:
        drop_train.add(i)
    elif i < n_tv:
        drop_val.add(i - n_train)

for key in ["exact_duplicates", "near_duplicates"]:
    for s in lab.info[key]["sets"]:
        s = sorted(s)
        if s[-1] >= n_tv:                  # Gruppe enthält ein Test-Bild
            for i in s:
                if i < n_tv:
                    mark(i)
        else:                              # nur Train/Val
            for i in s[1:]:
                mark(i)

# nach Sichtprüfung, falls kaputt:
# imagelab.visualize(issue_types=["dark", "low_information"])
# drop_train.update(int(i) for i in imagelab.issues.query("is_dark_issue or is_low_information_issue").index)

DROP_FILE.write_text(json.dumps({"seed": 67, "train": sorted(drop_train), "val": sorted(drop_val)}))
print(len(drop_train), "aus Train,", len(drop_val), "aus Val entfernt")

/Users/adrianuhe/Studium/dlcv/deep-learning-computer-vision/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 